# LDaCA Wordflow: Notebook Launcher

This notebook starts the packaged LDaCA Wordflow frontend and backend as one loopback server inside Binder.

The launch cell below will:
1. Start Wordflow on `127.0.0.1:8001`.
2. Let Jupyter Server Proxy expose it below this notebook's authenticated URL.
3. Display a clickable link to open the web app.

> **Storage:** Binder storage is session-scoped. The backend defaults to `DATA_ROOT=~/Documents/ldaca`; set `DATA_ROOT` before launching if this BinderHub provides a persistent mount.

In [ ]:
import os

from IPython.display import Markdown, display
from ldaca_wordflow import start_async_server

from utils import configure_hub_networking

PORT = 8001

# Allow the public BinderHub host before Wordflow loads its security settings.
await configure_hub_networking()

# Wordflow derives its ASGI root_path from JUPYTERHUB_SERVICE_PREFIX and
# returns only after application startup completes.
wordflow_server = await start_async_server(port=PORT)
proxy_path = (
    f"{os.environ['JUPYTERHUB_SERVICE_PREFIX'].rstrip('/')}"
    f"/proxy/{wordflow_server.settings.backend_port}/"
)
display(Markdown(f"# [Open LDaCA Wordflow]({proxy_path})"))

### Inspecting the stored data

- Since v0.7, workspace state is owned by the running server (the old module-level `workspace_manager` no longer exists), so this cell inspects what the server has persisted on disk instead
- User files and imports live under `DATA_ROOT/users/<user>/`; workspace snapshots live under `DATA_ROOT/workspaces/`
- If this is your first run, the folders are created automatically when the backend starts

In [ ]:
# The ServerHandle from the launcher cell carries the resolved settings.
data_root = wordflow_server.settings.get_data_root()
print(f"Data root: {data_root}\n")

workspaces = [
    p for p in sorted((data_root / "workspaces").glob("*")) if not p.name.startswith(".")
]
if not workspaces:
    print("No workspaces exist yet — create one in the app.")
else:
    print("Workspaces:")
    for w in workspaces:
        print(f"  - {w.name}")

users_root = data_root / "users"
if users_root.is_dir():
    print("\nUsers:", ", ".join(p.name for p in sorted(users_root.iterdir()) if p.is_dir()))

### Stop Wordflow

Run the cell below before restarting Wordflow on the same port or when you are finished with the Binder session.

In [ ]:
await wordflow_server.close()